In [ ]:
import glob, gzip, json, numpy as np, joblib
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, GroupKFold
files = sorted(glob.glob("/kaggle/input/**/mined_candidates.jsonl.gz", recursive=True))
print("chunk files:", files)
X, y, grp, src = [], [], [], []
seen = set()
for fpath in files:
    with gzip.open(fpath, "rt") as f:
        for line in f:
            r = json.loads(line)
            k = (r["src"], r["dataset"], r["t"], tuple(round(v,1) for v in r["p"]), tuple(round(v,1) for v in r["c1"]), tuple(round(v,1) for v in r["c2"]))
            if k in seen: continue
            seen.add(k); X.append(r["f"]); y.append(int(r["label"])); grp.append(r["dataset"]); src.append(r["src"])
X = np.array(X, dtype=np.float32); y = np.array(y); grp = np.array(grp); src = np.array(src)
print(f"merged: {len(y)} candidates, {int(y.sum())} positives, {len(set(grp))} videos")
for s in ("safediv", "fork"): m = src == s; print(f"  {s}: n={int(m.sum())} pos={int(y[m].sum())}")
FEATURES = ["pd1","pd2","sister_dist","pd_asym","divergence","local_density","par_mean","par_max","par_std","c1_mean","c1_max","c1_std","c2_mean","c2_max","c2_std","mid_dist","par_speed","cosang","dens_t0","int_ratio","int_sym","zdiff"]
# subsample negatives 10:1 per video-group to keep training fast and balanced
rng = np.random.default_rng(0)
pos_idx = np.where(y == 1)[0]; neg_idx = np.where(y == 0)[0]
neg_keep = rng.choice(neg_idx, size=min(len(neg_idx), max(20000, 200 * len(pos_idx))), replace=False)
idx = np.concatenate([pos_idx, neg_keep]); Xs, ys, gs = X[idx], y[idx], grp[idx]
print("train subset:", len(ys), "pos", int(ys.sum()))
clf = HistGradientBoostingClassifier(max_iter=250, learning_rate=0.05, max_depth=3, min_samples_leaf=20, l2_regularization=3.0, class_weight="balanced", random_state=0)
gkf = GroupKFold(n_splits=5)
aucs = cross_val_score(clf, Xs, ys, groups=gs, cv=gkf, scoring="roc_auc")
print("GroupKFold(video) CV AUC:", np.round(aucs, 4), "mean:", float(aucs.mean()))
# precision at recall on out-of-fold predictions
from sklearn.model_selection import cross_val_predict
p = cross_val_predict(clf, Xs, ys, groups=gs, cv=gkf, method="predict_proba")[:, 1]
for thr in (0.3, 0.5, 0.7, 0.85):
    sel = p >= thr; tp = int((sel & (ys == 1)).sum()); fp = int((sel & (ys == 0)).sum()); fn = int((~sel & (ys == 1)).sum())
    print(f"  thr {thr}: TP {tp} FP {fp} FN {fn} -> precision {tp/max(1,tp+fp):.3f} recall {tp/max(1,tp+fn):.3f}  (FP rate on full neg set ≈ {fp/len(neg_keep)*len(neg_idx):.0f})")
clf.fit(Xs, ys); joblib.dump(clf, "/kaggle/working/mitosis_gate_v3m.joblib")
json.dump({"features": FEATURES, "cv_auc": float(aucs.mean()), "n": int(len(y)), "pos": int(y.sum()), "chunks": files}, open("/kaggle/working/mitosis_gate_v3m_meta.json", "w"))
print("saved merged gate v3m")
